# KAN-Transformer for Chest X-Ray Pneumonia Classification

This notebook builds a beginner-friendly Vision Transformer (ViT)-style classifier for **binary chest X-ray classification** (**NORMAL** vs **PNEUMONIA**).

Main experiment idea:
- Keep the Transformer architecture standard (attention, LayerNorm, residuals, GELU).
- Replace **only the first FFN Linear layer** with a **pykan-based KAN wrapper**.

## 1) Install / import dependencies

In [ ]:
# If needed, uncomment and run:
# !pip install torch torchvision scikit-learn matplotlib seaborn pandas kagglehub pykan

import os
import json
import copy
from dataclasses import dataclass, asdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
    roc_auc_score,
)

import kagglehub
from kan import KAN

print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

## 2) Download dataset using KaggleHub

In [ ]:
path = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")
print("Path to dataset files:", path)

## 3) Dataset path inspection

In [ ]:
dataset_root = Path(path)

# KaggleHub may place the actual data under an inner folder like 'chest_xray'
candidate_root = dataset_root / 'chest_xray'
if candidate_root.exists():
    data_root = candidate_root
else:
    data_root = dataset_root

print('Using data_root:', data_root)
print('Top-level contents:')
for p in sorted(data_root.iterdir()):
    print(' -', p.name)

for split in ['train', 'val', 'test']:
    split_path = data_root / split
    if split_path.exists():
        print(f"\n{split} classes:")
        for cls in sorted(split_path.iterdir()):
            if cls.is_dir():
                count = len(list(cls.glob('*')))
                print(f"  {cls.name}: {count}")

## 4) Data transforms and dataloaders

In [ ]:
@dataclass
class Config:
    # Data
    image_size: int = 224
    batch_size: int = 16
    num_workers: int = 2

    # Model
    patch_size: int = 16
    in_channels: int = 3
    embed_dim: int = 128
    num_heads: int = 4
    num_layers: int = 3
    mlp_ratio: int = 4
    num_classes: int = 2

    # KAN (pykan)
    kan_grid: int = 3
    kan_k: int = 3  # cubic

    # Training
    learning_rate: float = 1e-4
    max_epochs: int = 40
    min_epochs: int = 30
    patience: int = 5

cfg = Config()
print(cfg)

# Chest X-ray images are often grayscale, convert to RGB (3 channels).
train_transform = transforms.Compose([
    transforms.Resize((cfg.image_size, cfg.image_size)),
    transforms.Grayscale(num_output_channels=3),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
])

val_test_transform = transforms.Compose([
    transforms.Resize((cfg.image_size, cfg.image_size)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
])

train_dataset = datasets.ImageFolder(data_root / 'train', transform=train_transform)
val_dataset = datasets.ImageFolder(data_root / 'val', transform=val_test_transform)
test_dataset = datasets.ImageFolder(data_root / 'test', transform=val_test_transform)

train_loader = DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers, pin_memory=True)

class_names = train_dataset.classes
print('Classes:', class_names)
print('Train size:', len(train_dataset))
print('Val size:', len(val_dataset))
print('Test size:', len(test_dataset))

## 5) Model configuration

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

# Helpful class index mapping
class_to_idx = train_dataset.class_to_idx
idx_to_class = {v: k for k, v in class_to_idx.items()}
print('class_to_idx:', class_to_idx)

## 6) KANLinearWrapper implementation

In [ ]:
class KANLinearWrapper(nn.Module):
    """
    Wrapper around pykan.KAN for Transformer FFN usage.

    Input:  [batch, tokens, dim]
    Flatten to [batch*tokens, dim]
    Apply KAN
    Output: [batch, tokens, hidden_dim]
    """
    def __init__(self, in_dim: int, out_dim: int, grid: int = 3, k: int = 3):
        super().__init__()
        self.in_dim = in_dim
        self.out_dim = out_dim

        # Simple KAN mapping: in_dim -> out_dim
        self.kan = KAN(width=[in_dim, out_dim], grid=grid, k=k)
        if hasattr(self.kan, 'speed'):
            self.kan.speed()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, t, d = x.shape
        if d != self.in_dim:
            raise ValueError(f'Expected dim {self.in_dim}, got {d}')

        x_flat = x.reshape(b * t, d)
        y_flat = self.kan(x_flat)
        y = y_flat.reshape(b, t, self.out_dim)
        return y

## 7) Patch embedding implementation

In [ ]:
class PatchEmbed(nn.Module):
    """Split image into non-overlapping patches, flatten, and project to embedding dim."""
    def __init__(self, image_size: int, patch_size: int, in_channels: int, embed_dim: int):
        super().__init__()
        assert image_size % patch_size == 0, 'patch_size must divide image_size'
        self.image_size = image_size
        self.patch_size = patch_size
        self.in_channels = in_channels
        self.num_patches_per_side = image_size // patch_size
        self.num_patches = self.num_patches_per_side ** 2
        self.patch_dim = in_channels * patch_size * patch_size
        self.proj = nn.Linear(self.patch_dim, embed_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, C, H, W]
        b, c, _, _ = x.shape
        p = self.patch_size

        patches = x.unfold(2, p, p).unfold(3, p, p)
        # [B, C, nH, nW, p, p] -> [B, num_patches, patch_dim]
        patches = patches.contiguous().view(b, c, -1, p, p)
        patches = patches.permute(0, 2, 1, 3, 4).contiguous()
        patches = patches.view(b, self.num_patches, -1)
        return self.proj(patches)

## 8) Transformer encoder block with KAN FFN

In [ ]:
class TransformerEncoderBlock(nn.Module):
    def __init__(self, embed_dim: int, num_heads: int, mlp_ratio: int, cfg: Config):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads, batch_first=True)

        self.norm2 = nn.LayerNorm(embed_dim)
        hidden_dim = embed_dim * mlp_ratio

        # IMPORTANT: replace only the FIRST FFN Linear with KAN wrapper
        self.fc1 = KANLinearWrapper(embed_dim, hidden_dim, grid=cfg.kan_grid, k=cfg.kan_k)

        # Keep activation + second layer standard
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_dim, embed_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Attention block (standard)
        x_norm = self.norm1(x)
        attn_out, _ = self.attn(x_norm, x_norm, x_norm, need_weights=False)
        x = x + attn_out

        # FFN block (first layer replaced by KAN)
        y = self.norm2(x)
        y = self.fc1(y)
        y = self.act(y)
        y = self.fc2(y)
        x = x + y
        return x

## 9) Full KAN-ViT model

In [ ]:
class KANViT(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg

        self.patch_embed = PatchEmbed(
            image_size=cfg.image_size,
            patch_size=cfg.patch_size,
            in_channels=cfg.in_channels,
            embed_dim=cfg.embed_dim,
        )

        num_patches = self.patch_embed.num_patches

        self.cls_token = nn.Parameter(torch.zeros(1, 1, cfg.embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, cfg.embed_dim))

        self.blocks = nn.ModuleList([
            TransformerEncoderBlock(cfg.embed_dim, cfg.num_heads, cfg.mlp_ratio, cfg)
            for _ in range(cfg.num_layers)
        ])

        self.norm = nn.LayerNorm(cfg.embed_dim)
        self.head = nn.Linear(cfg.embed_dim, cfg.num_classes)

        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b = x.size(0)
        x = self.patch_embed(x)

        cls_tokens = self.cls_token.expand(b, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)
        x = x + self.pos_embed

        for blk in self.blocks:
            x = blk(x)

        x = self.norm(x)
        cls_rep = x[:, 0]
        logits = self.head(cls_rep)
        return logits

model = KANViT(cfg).to(device)
print(model.__class__.__name__)

## 10) Training and validation functions

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        bs = images.size(0)
        total_loss += loss.item() * bs
        total += bs

    return total_loss / max(total, 1)

@torch.no_grad()
def evaluate_loss_acc(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        logits = model(images)
        loss = criterion(logits, labels)

        bs = images.size(0)
        total_loss += loss.item() * bs
        total += bs

        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()

    return total_loss / max(total, 1), correct / max(total, 1)

## 11) Early stopping and checkpoint saving

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=cfg.learning_rate)
criterion = nn.CrossEntropyLoss()

os.makedirs('results', exist_ok=True)
checkpoint_path = os.path.join('results', 'best_kan_transformer_chest_xray.pt')

best_val_loss = float('inf')
best_state = None
best_epoch = -1
no_improve = 0

history = []

## 12) Training loop

In [ ]:
for epoch in range(1, cfg.max_epochs + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc = evaluate_loss_acc(model, val_loader, criterion, device)

    history.append({
        'epoch': epoch,
        'train_loss': float(train_loss),
        'val_loss': float(val_loss),
        'val_acc': float(val_acc),
    })

    print(f"Epoch {epoch:02d}/{cfg.max_epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc*100:.2f}%")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = copy.deepcopy(model.state_dict())
        best_epoch = epoch
        no_improve = 0
        torch.save(best_state, checkpoint_path)
    else:
        no_improve += 1

    # minimum 30 epochs before early stopping
    if epoch >= cfg.min_epochs and no_improve >= cfg.patience:
        print(f"Early stopping at epoch {epoch} (patience={cfg.patience}).")
        break

if best_state is not None:
    model.load_state_dict(best_state)

print('Best epoch:', best_epoch)
print('Best val loss:', best_val_loss)
print('Best checkpoint saved to:', checkpoint_path)

## 13) Test evaluation

In [ ]:
@torch.no_grad()
def predict_with_probs(model, loader, device):
    model.eval()
    all_y = []
    all_pred = []
    all_prob = []

    for images, labels in loader:
        images = images.to(device)
        logits = model(images)
        probs = F.softmax(logits, dim=1)
        preds = probs.argmax(dim=1)

        all_y.append(labels.cpu().numpy())
        all_pred.append(preds.cpu().numpy())
        all_prob.append(probs.cpu().numpy())

    y_true = np.concatenate(all_y)
    y_pred = np.concatenate(all_pred)
    y_prob = np.concatenate(all_prob)
    return y_true, y_pred, y_prob

y_true, y_pred, y_prob = predict_with_probs(model, test_loader, device)

acc = accuracy_score(y_true, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary', zero_division=0)
cm = confusion_matrix(y_true, y_pred)
report = classification_report(y_true, y_pred, target_names=class_names, digits=4, zero_division=0)

# ROC-AUC for binary classification using probability of PNEUMONIA class
pneumonia_idx = class_to_idx['PNEUMONIA']
roc_auc = roc_auc_score(y_true, y_prob[:, pneumonia_idx])

print('\n===== Test Metrics =====')
print(f'Accuracy : {acc:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall   : {recall:.4f}')
print(f'F1-score : {f1:.4f}')
print(f'ROC-AUC  : {roc_auc:.4f}')
print('\nConfusion Matrix:\n', cm)
print('\nClassification Report:\n', report)

## 14) Save results

In [ ]:
# Save training history CSV
history_df = pd.DataFrame(history)
history_csv_path = os.path.join('results', 'training_history.csv')
history_df.to_csv(history_csv_path, index=False)

# Save confusion matrix PNG
cm_png_path = os.path.join('results', 'confusion_matrix.png')
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix - KAN Transformer Chest X-Ray')
plt.tight_layout()
plt.savefig(cm_png_path, dpi=150)
plt.show()

# Save classification report TXT
report_txt_path = os.path.join('results', 'classification_report.txt')
with open(report_txt_path, 'w') as f:
    f.write(report)

# Save metrics JSON
metrics = {
    'accuracy': float(acc),
    'precision': float(precision),
    'recall': float(recall),
    'f1_score': float(f1),
    'roc_auc': float(roc_auc),
    'best_epoch': int(best_epoch),
    'best_val_loss': float(best_val_loss),
    'checkpoint_path': checkpoint_path,
    'config': asdict(cfg),
}
metrics_json_path = os.path.join('results', 'final_metrics.json')
with open(metrics_json_path, 'w') as f:
    json.dump(metrics, f, indent=2)

print('Saved files:')
print(' -', checkpoint_path)
print(' -', history_csv_path)
print(' -', metrics_json_path)
print(' -', cm_png_path)
print(' -', report_txt_path)

## 15) Final conclusion

This notebook demonstrates a simple KAN-ViT experiment for chest X-ray pneumonia detection:
- ViT backbone remains standard.
- Only the first FFN linear layer is replaced by `KANLinearWrapper` using **pykan**.
- The workflow includes training, early stopping, checkpointing, full evaluation, and result export.

You can now tune hyperparameters in `Config` (patch size, embed dim, layers, learning rate, etc.) and compare against a baseline ViT with `nn.Linear` in the same location.